# HBICP iteration 3 on Colab
Runtime > Change runtime type > GPU (T4). Run the cells in order. If Colab disconnects, reopen and run all again: it resumes from `last.pt` on Drive.

Runs (all with the residual anchor o_T + dec(z_k) - dec(z_0), except the reference):
- `n8_ref`: reference, 8000 steps (already done)
- `n8_resid`: residual anchor, 20000 steps
- `n8_resid_all`: + variable horizon H in [16, 64] + per-k-normalised loss, 20000 steps
- `n8_resid_big`: same as `resid_all`, 30000 steps on 5000 training Hamiltonians (`train5k.h5`)

Then: train-vs-test k* diagnostic, and the fair HL baseline **rerun** (the previous `hl_fair` numbers came from a pair-index bug in `ehrenfest_system`, fixed; old results are discarded). Evaluation goes up to T = 127. Quadratic readout off.

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
REPO = 'https://github.com/Alessio-Pecilli/Hamiltonian-Blind-In-Context-Propagator.git'
if not os.path.exists('/content/hbicp'):
    subprocess.run(['git', 'clone', REPO, '/content/hbicp'], check=True)
else:
    subprocess.run(['git', '-C', '/content/hbicp', 'pull'], check=True)
%cd /content/hbicp/Hamiltonian-Blind-In-Context-Propagator
!pip install -q h5py pyyaml scipy
# datasets, checkpoints and results live on Drive so they survive disconnects
D = '/content/drive/MyDrive/hbicp'
for d in ['datasets', 'runs', 'results']:
    os.makedirs(f'{D}/{d}', exist_ok=True)
    if not os.path.islink(d): os.symlink(f'{D}/{d}', d)
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/hbicp/Hamiltonian-Blind-In-Context-Propagator
True Tesla T4


In [ ]:
# new code paths (residual anchor, per-k loss) must pass before spending GPU time
!python -m pytest -q tests

In [5]:
# datasets (skipped if already on Drive)
!test -f datasets/n8/test.h5 || bash scripts/make_datasets.sh local 2
!test -f datasets/n8/train5k.h5 || bash scripts/make_datasets.sh local5k 2

In [6]:
# reference, residual anchor, + variable horizon + per-k loss, + more data; idempotent
import os, subprocess, torch
RES = ['model.residual=true']
ALL = RES + ['train.var_horizon=true', 'train.H=64', 'train.H_min=16', 'loss.per_k_norm=true']
runs = [('n8_ref', 8000, []), ('n8_resid', 20000, RES), ('n8_resid_all', 20000, ALL),
        ('n8_resid_big', 30000, ALL + ['data.train=datasets/n8/train5k.h5'])]
# subset to run now; set ONLY = None for all. n8_resid_big: only after reading the train-vs-test k* diagnostic
ONLY = ['n8_resid', 'n8_resid_all']
def step(d):
    p = f'{d}/last.pt'
    return torch.load(p, map_location='cpu', weights_only=False)['step'] if os.path.exists(p) else 0
for name, steps, extra in runs:
    if ONLY and name not in ONLY: continue
    d, res = f'runs/{name}', f'results/{name}/results.json'
    if os.path.exists(res): continue
    os.makedirs(d, exist_ok=True)
    if step(d) < steps:
        a = ['python', 'train.py', '--preset', 'local', '--set', f'train.steps={steps}', 'train.seed=0', f'train.out_dir={d}'] + extra
        if os.path.exists(f'{d}/last.pt'): a += ['--resume', f'{d}/last.pt']
        import shlex  # full output (stderr included) also goes to runs/<name>/train.log on Drive
        subprocess.run(['bash', '-o', 'pipefail', '-c', ' '.join(map(shlex.quote, a)) + f' 2>&1 | tee -a {d}/train.log'], check=True)
    if step(d) >= steps:
        subprocess.run(['python', '-m', 'eval.evaluate', '--ckpt', f'{d}/last.pt', '--data', 'datasets/n8/test.h5',
                        '--T', '8', '16', '32', '64', '96', '127', '--horizon', '64',
                        '--baselines', 'persistence', 'dmd', 'dmd4', 'oracle', 'hl_oracle', '--out', f'results/{name}'], check=True)

In [ ]:
# k* on training vs test windows: train >> test = dataset too small (overfit); train ~ test and both low = model undertrained / too small
import json, subprocess
for name in ['n8_ref', 'n8_resid', 'n8_resid_all', 'n8_resid_big']:
    if not os.path.exists(f'runs/{name}/last.pt'): continue
    out = f'results/{name}_traintest'
    for split in ['train', 'test']:
        if not os.path.exists(f'{out}/{split}/results.json'):
            subprocess.run(['python', '-m', 'eval.evaluate', '--ckpt', f'runs/{name}/last.pt', '--data', f'datasets/n8/{split}.h5',
                            '--T', '8', '16', '32', '64', '--horizon', '64', '--baselines', '--max_traj', '64', '--out', f'{out}/{split}'], check=True)
    for split in ['train', 'test']:
        r = next(iter(json.load(open(f'{out}/{split}/results.json')).values()))

## Fair HL baseline
Generic 2-local Hamiltonian fitted on the context only (`hl_fair_o0`: psi_0 from o_0; `hl_fair_fit`: psi_0 fitted). CPU, roughly 1 min per trajectory per T at n=8: lower `N_TRAJ` for a quick pass. Independent of the models; result goes to `results/hl_fair` and is added to the quad publish.

In [ ]:
# fair HL baseline on the product test set; rerun once after the pair-index bugfix (old results discarded)
import os, shutil, subprocess
N_TRAJ = 20
if not os.path.exists('results/hl_fair/.fixed'):
    shutil.rmtree('results/hl_fair', ignore_errors=True)
    subprocess.run(['python', '-m', 'eval.evaluate', '--data', 'datasets/n8/test.h5', '--T', '16', '32', '64', '--horizon', '64',
                    '--baselines', 'hl_fair_o0', 'hl_fair_fit', '--max_traj', str(N_TRAJ), '--out', 'results/hl_fair'], check=True)
    open('results/hl_fair/.fixed', 'w').close()

## Publish
One-time setup: GitHub > Settings > Developer settings > Fine-grained tokens > only this repo, Contents: Read and write. In Colab, left sidebar key icon > Add secret `GH_TOKEN` with the token, enable Notebook access.

In [7]:
# publish to the dashboard (git push) from Colab; needs Colab secret GH_TOKEN (see markdown above)
import os, shutil, subprocess
from google.colab import userdata
tok = userdata.get('GH_TOKEN')
subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://{tok}@github.com/Alessio-Pecilli/Hamiltonian-Blind-In-Context-Propagator.git'], check=True)
subprocess.run(['git', 'config', 'user.name', 'Alessio Pecilli'], check=True)
subprocess.run(['git', 'config', 'user.email', 'alessiopecilli@gmail.com'], check=True)
jobs = [('n8_ref', ['n8_ref'], 'n8 A riferimento (no probe, no opzioni coppie, S off)',
         'Run A: riferimento. pair_attn/triangle/pair_latent spenti, nessun L_probe, use_S=false, n=8, 8000 step, seed 0, validazione ogni 1000 step.'),
        ('n8_resid', ['n8_ref', 'n8_resid'], 'n8 residual anchor, 20k step',
         'Residual anchor o_T + dec(z_k) - dec(z_0): l errore va a 0 per k -> 0. 20000 step (A: 8000), 1000 Hamiltoniane, seed 0.'),
        ('n8_resid_all', ['n8_ref', 'n8_resid', 'n8_resid_all'], 'n8 residual + orizzonte variabile + loss per-k, 20k step',
         'Residual + orizzonte variabile (H uniforme in [16, 64]) + loss normalizzata per k (errore della persistenza). 20000 step, 1000 Hamiltoniane.'),
        ('n8_resid_big', ['n8_ref', 'n8_resid_all', 'n8_resid_big', 'hl_fair'], 'n8 residual completo, 5000 Hamiltoniane, 30k step (+ HL equo corretto)',
         'Come resid_all ma 30000 step su 5000 Hamiltoniane di training (valutazione fino a T = 127). Baseline HL equa ricalcolata dopo la correzione dell indice delle coppie in ehrenfest_system.')]
for name, keep, label, notes in jobs:
    marker = f'results/{name}/.published'
    if os.path.exists(marker): print(name, 'skip: gia pubblicato (.published)'); continue
    if not os.path.exists(f'results/{name}/results.json'): print(name, 'skip: manca results/%s/results.json' % name); continue
    shutil.rmtree('stage_publish', ignore_errors=True)
    for k in keep:
        if not os.path.exists(f'results/{k}/results.json'): continue  # e.g. hl_fair not computed yet
        os.makedirs(f'stage_publish/runs/{k}', exist_ok=True); os.makedirs(f'stage_publish/results/{k}')
        if k != 'hl_fair':
            for f in ['config.json', 'log.jsonl', 'last.pt']: shutil.copy(f'runs/{k}/{f}', f'stage_publish/runs/{k}')
        shutil.copy(f'results/{k}/results.json', f'stage_publish/results/{k}')
    subprocess.run(['git', 'pull', '--rebase', '-q'], check=True)
    r = subprocess.run(['python', '-m', 'tools.publish_dashboard', '--label', label, '--notes', notes, '--runs', 'stage_publish/runs',
        '--results', 'stage_publish/results', '--data', 'datasets/n8/test.h5', '--ckpt', f'runs/{name}/last.pt',
        '--T', '32', '--horizon', '64', '--sample', '0', '--keep', '100'])
    if r.returncode == 0: open(marker, 'w').close()
    shutil.rmtree('stage_publish', ignore_errors=True)
print('done')
